In [0]:
%run ../logging_utils

In [0]:
import json
import ast
import logging
from datetime import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
from dateutil.relativedelta import relativedelta

In [0]:
from datetime import datetime

RunId = dbutils.widgets.get("RunId")
cod_workflowName = dbutils.widgets.get("cod_workflowName")
cod_pipelineID = dbutils.widgets.get("cod_pipelineID")
cod_layerName = dbutils.widgets.get("cod_layerName")
cod_productName = dbutils.widgets.get("cod_productName")
snapshot_date = dbutils.widgets.get("snapshot_date")
cod_flowName = dbutils.widgets.get("cod_flowName")
env = dbutils.widgets.get("env")
input_params=dbutils.widgets.get("input_params")

general_utils = LoggingUtils(spark)
logger = general_utils.set_logging()

config = general_utils.load_config(general_utils.get_variable(cod_productName)['config_setup_path'])
snapshot_date = general_utils.date_to_start_month(snapshot_date)
catalog_name = config['environment'][env]['catalog']
schema_name = config['environment'][env]['schema']

logger.info(f"Snapshot_date materialized: {snapshot_date}")

try:
    logger.info(f"Log generation process INITIALIZING")
    # logger.info(f"Get parameters")
    # params = general_utils.get_pipeline_parameters()

    logger.info(f"Obtaining log processor reference")
    log_catalog, log_database, log_table = general_utils.get_log_processor_reference(
        catalog_name, 
        "monitoring", 
        config['logging']['processor']['table']
    )
    
    full_table_path = f"{log_catalog}.{log_database}.{log_table}"

except Exception as e:
    logger.error(f"Log generation process ERROR as {e}")
    raise

try:
    logger.info(f"Log generation process START")
    
    now_date = datetime.now().strftime("%Y-%m-%d")
    now_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    def sql_val(val):
        return f"'{val}'" if val is not None and str(val).strip() != "" else "NULL"

    sql_query = f"""
        INSERT INTO {full_table_path} (
            cod_workflowName, cod_layerName, dat_processDate, dat_processStartDate, 
            dat_processEndDate, cod_pipelineID, cod_runID, cod_productname, 
            cod_flowName, cod_flowNameSub, dat_snapshotDate, cod_processStatus, 
            cod_processErrorCode, des_processErrorDes
        ) VALUES (
            {sql_val(cod_workflowName)}, 
            {sql_val(cod_layerName)}, 
            CAST({sql_val(now_date)} AS DATE), 
            CAST({sql_val(now_timestamp)} AS TIMESTAMP), 
            NULL, 
            {sql_val(cod_pipelineID)}, 
            {sql_val(RunId)}, 
            {sql_val(cod_productName)}, 
            {sql_val(cod_flowName)}, 
            NULL, 
            CAST({sql_val(snapshot_date)} AS DATE), 
            'Started', 
            NULL, 
            NULL
        )
    """

    logger.info(f"Writing log via SQL on {full_table_path}")
    spark.sql(sql_query)
    
    logger.info(f"Log generation process END")

except Exception as e:
    logger.error(f"Log generation process ERROR as {e}")
    raise

In [0]:
dbutils.notebook.exit("Success!!")